# Notebook 01 — Data Cleaning

**Purpose**: Load all raw datasets, standardize formats, merge onto a master institution lookup table, and export `data/processed/korean_students_master.csv`.

## Cleaning Decisions Log

*Document every non-trivial cleaning decision here as you make it. Example:*
- `2024-04-10`: Standardized year format from 'YYYY/YY' → integer starting year
- `2024-04-10`: Dropped 3 IIE rows where institution name could not be matched to IPEDS (listed below)
- ...

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────
RAW   = Path('../data/raw')
PROC  = Path('../data/processed')
PROC.mkdir(exist_ok=True)

print('Raw files available:', list(RAW.glob('*')))

## 1. Load IIE Open Doors — All Places of Origin

Filter to South Korea rows only. Standardize year format to integer.

In [ ]:
# Load IIE All Places of Origin
# Expected columns: Country, [Year columns as '2000/01', '2001/02', ...]
iie_origin_path = RAW / 'iie_open_doors_korea.csv'

# TODO: Download from https://opendoorsdata.org/data/international-students/all-places-of-origin/
# and save as data/raw/iie_open_doors_korea.csv

if iie_origin_path.exists():
    iie_raw = pd.read_csv(iie_origin_path)
    # Filter to South Korea
    korea = iie_raw[iie_raw['Place of Origin'].str.contains('Korea, South|South Korea', case=False, na=False)].copy()
    # Melt wide → long
    korea_long = korea.melt(id_vars=['Place of Origin'], var_name='academic_year', value_name='korean_students')
    # Standardize year: '2020/21' → 2020
    korea_long['year'] = korea_long['academic_year'].str.extract(r'(\d{4})').astype(int)
    korea_long['korean_students'] = pd.to_numeric(korea_long['korean_students'].astype(str).str.replace(',',''), errors='coerce')
    print(korea_long.head())
    print(f'Years: {sorted(korea_long.year.unique())}')
else:
    print('⚠️  File not yet downloaded. See data/README.md for source URL.')

## 2. Load IIE Open Doors — Leading Institutions

Filter to South Korea rows. This is the institution-level enrollment data.

In [ ]:
inst_path = RAW / 'iie_leading_institutions.csv'

if inst_path.exists():
    iie_inst = pd.read_csv(inst_path)
    # TODO: Inspect columns and filter to Korea rows
    print(iie_inst.columns.tolist())
    print(iie_inst.head())
else:
    print('⚠️  File not yet downloaded. See data/README.md.')

## 3. Load IPEDS — Institutional Characteristics

Build master institution lookup: name, IPEDS unit ID, state, type, tuition.

In [ ]:
ipeds_path = RAW / 'ipeds_inst_characteristics.csv'

if ipeds_path.exists():
    ipeds = pd.read_csv(ipeds_path, encoding='latin-1')
    # Select relevant columns
    cols_keep = ['UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'CARNEGIE', 'TUITION2', 'TUITION3']
    cols_keep = [c for c in cols_keep if c in ipeds.columns]
    ipeds_clean = ipeds[cols_keep].copy()
    # Map control codes
    control_map = {1: 'Public', 2: 'Private nonprofit', 3: 'Private for-profit'}
    if 'CONTROL' in ipeds_clean.columns:
        ipeds_clean['control'] = ipeds_clean['CONTROL'].map(control_map)
    print(ipeds_clean.shape)
    print(ipeds_clean.head())
else:
    print('⚠️  File not yet downloaded. See data/README.md.')

## 4. Institution Name Matching (IIE ↔ IPEDS)

Fuzzy match institution names. Flag low-confidence matches for manual review.

In [ ]:
# pip install rapidfuzz --break-system-packages
# from rapidfuzz import process, fuzz

# TODO: After both datasets are loaded, run fuzzy matching
# Example:
#
# iie_names   = iie_inst['Institution'].dropna().unique().tolist()
# ipeds_names = ipeds_clean['INSTNM'].dropna().unique().tolist()
#
# results = []
# for name in iie_names:
#     match, score, idx = process.extractOne(name, ipeds_names, scorer=fuzz.token_sort_ratio)
#     results.append({'iie_name': name, 'ipeds_match': match, 'score': score, 'unitid': ipeds_clean.iloc[idx]['UNITID']})
#
# match_df = pd.DataFrame(results)
# low_conf = match_df[match_df['score'] < 90]
# print(f'Low-confidence matches requiring manual review: {len(low_conf)}')
# low_conf.to_csv(PROC / 'name_match_review.csv', index=False)
print('Name matching step — run after both datasets are downloaded.')

## 5. Load QS Rankings

In [ ]:
qs_path = RAW / 'qs_rankings.csv'

if qs_path.exists():
    qs = pd.read_csv(qs_path)
    print(qs.columns.tolist())
    print(qs.head())
else:
    print('⚠️  Download from Kaggle: akashbommidi/2026-qs-world-university-rankings')

## 6. Build Master Dataset & Export

In [ ]:
# TODO: After all files are downloaded and matched:
# 1. Merge IIE institution-level data with IPEDS lookup
# 2. Merge QS rank
# 3. Merge Census Korean-American population by state
# 4. Export

# master = (
#     iie_inst_clean
#     .merge(institution_lookup, on='institution_name', how='left')
#     .merge(qs_clean[['institution_name','qs_rank']], on='institution_name', how='left')
#     .merge(census_state[['state','year','korean_pop']], on=['state','year'], how='left')
# )

# master.to_csv(PROC / 'korean_students_master.csv', index=False)
# print(f'Exported master dataset: {master.shape}')
print('Export step — run after all source data is downloaded and merged.')